In [1]:
# Cell 0 — Drive Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, time, json, glob, warnings, urllib.request
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve,
)

warnings.filterwarnings('ignore')
RANDOMSTATE = 42
np.random.seed(RANDOMSTATE)

# Unified Drive paths
RUNID = 'ensemble_run_001'
DRIVEROOT = '/content/drive/MyDrive/tsad_ensemble_runs'
NOTEBOOKTAG = 'iforeststrict'

RUNDIR = os.path.join(DRIVEROOT, RUNID, NOTEBOOKTAG)
ARTIFACTDIR = os.path.join(RUNDIR, 'artifacts')
PREDICTIONSDIR = os.path.join(RUNDIR, 'predictions')
CACHEDIR = os.path.join(DRIVEROOT, '_cache')

for d in [ARTIFACTDIR, PREDICTIONSDIR, CACHEDIR]:
    os.makedirs(d, exist_ok=True)

print('RUNDIR       :', RUNDIR)
print('ARTIFACTDIR  :', ARTIFACTDIR)
print('PREDICTIONSDIR:', PREDICTIONSDIR)
print('CACHEDIR     :', CACHEDIR)


Mounted at /content/drive
RUNDIR       : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict
ARTIFACTDIR  : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/artifacts
PREDICTIONSDIR: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions
CACHEDIR     : /content/drive/MyDrive/tsad_ensemble_runs/_cache


In [2]:
# Cell 1 — Dataset Paths

MYDRIVE = '/content/drive/MyDrive'

CREDITCARDPATH = os.path.join(MYDRIVE, 'creditcard.csv')

NAB_CANDIDATES = [
    os.path.join(MYDRIVE, 'NAB Dataset'),
    os.path.join(MYDRIVE, 'NAB'),
    os.path.join(MYDRIVE, 'datasets', 'NAB Dataset'),
    os.path.join(MYDRIVE, 'datasets', 'NAB'),
]

def resolve_nab_root(candidates):
    for c in candidates:
        if os.path.isdir(c):
            csvs = glob.glob(os.path.join(c, '**', '*.csv'), recursive=True)
            if len(csvs) > 0:
                return c
    return None

NABROOT = resolve_nab_root(NAB_CANDIDATES)

# Download NAB labels
NABLABELSLOCAL = os.path.join(CACHEDIR, 'nab_combined_windows.json')
NABLABELSURL = 'https://raw.githubusercontent.com/numenta/NAB/master/labels/combined_windows.json'

if not os.path.exists(NABLABELSLOCAL):
    print('Downloading NAB labels...')
    urllib.request.urlretrieve(NABLABELSURL, NABLABELSLOCAL)

with open(NABLABELSLOCAL) as f:
    NABWINDOWSMAP = json.load(f)

# Validation
assert os.path.exists(CREDITCARDPATH), f'creditcard.csv not found at {CREDITCARDPATH}'
assert NABROOT is not None, 'NAB root not found. Check Drive.'

nab_csvs = glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True)
nab_csvs = [f for f in nab_csvs if 'README' not in f]

print(f'CREDITCARDPATH : {CREDITCARDPATH}')
print(f'NABROOT        : {NABROOT}')
print(f'NAB CSV count  : {len(nab_csvs)}')
print(f'NAB labels     : {len(NABWINDOWSMAP)} entries, {sum(1 for v in NABWINDOWSMAP.values() if v)} with anomalies')


CREDITCARDPATH : /content/drive/MyDrive/creditcard.csv
NABROOT        : /content/drive/MyDrive/NAB Dataset
NAB CSV count  : 58
NAB labels     : 58 entries, 52 with anomalies


In [3]:
# Cell 2 — Shared Utilities

def robust_zscores(scores, eps=1e-6, clip=50.0):
    s = np.asarray(scores, dtype=float)
    med = np.median(s)
    mad = np.median(np.abs(s - med))
    return np.clip((s - med) / max(1.4826 * mad, eps), -clip, clip)

def keep_runs(y_pred, minlen=3):
    y = np.asarray(y_pred, dtype=int).copy()
    n = len(y)
    i = 0
    while i < n:
        if y[i] == 1:
            j = i
            while j < n and y[j] == 1:
                j += 1
            if j - i < minlen:
                y[i:j] = 0
            i = j
        else:
            i += 1
    return y

def point_adjust(y_true, y_pred):
    # PA protocol: if any point in a contiguous anomaly segment is detected,
    # credit the entire segment as detected
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_pred, dtype=int).copy()
    n = len(yt)
    i = 0
    while i < n:
        if yt[i] == 1:
            j = i
            while j < n and yt[j] == 1:
                j += 1
            if yp[i:j].any():
                yp[i:j] = 1
            i = j
        else:
            i += 1
    return yp

def temporal_slices(n, valfrac=0.30):
    n = int(n)
    v = int(max(1, round(n * valfrac)))
    v = min(v, n - 1)
    return slice(0, v), slice(v, n)

def best_strict_f1_threshold(y, s, ngrid=300, qlo=0.50, qhi=0.999, minrun=3):
    y = np.asarray(y, dtype=int)
    s = np.asarray(s, dtype=float)
    if y.sum() == 0:
        return float(np.max(s)) if len(s) else 0.0, 0.0
    qs = np.linspace(qlo, qhi, ngrid)
    thr_list = np.unique(np.quantile(s, qs))
    best_t, best_f = float(thr_list[-1]), -1.0
    for t in thr_list:
        p = keep_runs((s >= t).astype(int), minlen=minrun)
        f = f1_score(y, p, zero_division=0)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

def compute_metrics(y, pred, scores=None, prefix=''):
    m = {
        f'{prefix}precision': float(precision_score(y, pred, zero_division=0)),
        f'{prefix}recall': float(recall_score(y, pred, zero_division=0)),
        f'{prefix}f1': float(f1_score(y, pred, zero_division=0)),
    }
    if scores is not None and len(np.unique(y)) == 2:
        m[f'{prefix}rocauc'] = float(roc_auc_score(y, scores))
        m[f'{prefix}prauc'] = float(average_precision_score(y, scores))
    else:
        m[f'{prefix}rocauc'] = float('nan')
        m[f'{prefix}prauc'] = float('nan')
    return m

print('Utilities ready.')


Utilities ready.


In [4]:
# Cell 3 — Credit Card IF Agent

class CreditCardIFAgent:
    def __init__(self):
        self.model = None
        self.scaler = RobustScaler()
        self.threshold = None
        self.features = None

    def engineer_features(self, df):
        df = df.copy()
        # Time-based features
        df['hoursin'] = np.sin(2 * np.pi * df['Time'] / 86400.0)
        df['hourcos'] = np.cos(2 * np.pi * df['Time'] / 86400.0)
        # Amount features
        df['Amountlog'] = np.log1p(df['Amount'])
        df['AmountSq'] = df['Amount'] ** 2
        # V-feature interactions (top discriminative pairs from literature)
        df['V1xV2'] = df['V1'] * df['V2']
        df['V3xV4'] = df['V3'] * df['V4']
        df['V14xV17'] = df['V14'] * df['V17']
        df['V12xV14'] = df['V12'] * df['V14']
        # Absolute values of key V features (fraud tends to have extreme values)
        for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']:
            df[f'{v}_abs'] = df[v].abs()

        vcols = [f'V{i}' for i in range(1, 29)]
        extra = ['Amountlog', 'AmountSq', 'hoursin', 'hourcos',
                 'V1xV2', 'V3xV4', 'V14xV17', 'V12xV14',
                 'V1_abs', 'V3_abs', 'V4_abs', 'V7_abs',
                 'V10_abs', 'V12_abs', 'V14_abs', 'V17_abs']
        self.features = vcols + extra
        return df[self.features]

    def load(self):
        df = pd.read_csv(CREDITCARDPATH).dropna().reset_index(drop=True)
        y = df['Class'].astype(int).values
        X = self.engineer_features(df)
        return X, y

    def fit(self):
        Xall, yall = self.load()

        # Split: 80% tune, 20% test (same as Memto for alignment)
        idx = np.arange(len(yall), dtype=np.int64)
        idxtune, self.idxtest = train_test_split(
            idx, test_size=0.20, stratify=yall, random_state=RANDOMSTATE
        )

        # Fit scaler on normal samples from tune set
        Xtune = Xall.iloc[idxtune]
        ytune = yall[idxtune]
        Xnormal_tune = Xtune[ytune == 0]
        self.scaler.fit(Xnormal_tune)

        # Grid search contamination on tune set
        contamination_rates = [0.001, 0.002, 0.005, 0.01]
        best_cont, best_f1_tune = 0.001, -1.0

        for cont in contamination_rates:
            model = IsolationForest(
                n_estimators=400,
                contamination=cont,
                max_samples=min(50000, len(Xnormal_tune)),
                max_features=0.8,
                n_jobs=-1,
                random_state=RANDOMSTATE,
            )
            model.fit(self.scaler.transform(Xnormal_tune))

            scores_tune = -model.decision_function(self.scaler.transform(Xtune))
            # PR-curve threshold on tune set
            prec, rec, thresholds = precision_recall_curve(ytune, scores_tune)
            f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
            if len(f1s) > 0:
                best_idx = int(np.argmax(f1s))
                f1v = float(f1s[best_idx])
                if f1v > best_f1_tune:
                    best_f1_tune = f1v
                    best_cont = cont
                    self.model = model
                    self.threshold = float(thresholds[best_idx])

        print(f'Best contamination: {best_cont}, tune F1: {best_f1_tune:.4f}, threshold: {self.threshold:.6f}')
        return self

    def evaluate(self):
        Xall, yall = self.load()
        idxtest = self.idxtest

        Xtest = Xall.iloc[idxtest]
        ytest = yall[idxtest].astype(np.int8)
        scores = (-self.model.decision_function(self.scaler.transform(Xtest))).astype(np.float32)
        preds = (scores >= self.threshold).astype(np.int8)

        # Strict metrics
        strict = compute_metrics(ytest, preds, scores, prefix='')

        results = {
            'dataset': 'creditcard',
            'protocol': 'strict point-wise holdout (20% test)',
            **strict,
            'threshold': float(self.threshold),
            'n_anomalies_true': int(ytest.sum()),
            'n_anomalies_pred': int(preds.sum()),
        }

        return results, scores, ytest, preds, idxtest.astype(np.int64)

    def save(self):
        joblib.dump({
            'model': self.model,
            'scaler': self.scaler,
            'features': self.features,
            'threshold': self.threshold,
            'test_indices': self.idxtest,
        }, os.path.join(ARTIFACTDIR, 'ifcreditcard.joblib'))
        print('Saved CC artifact.')

ccagent = CreditCardIFAgent()
print('Fitting Credit Card IF...')
ccagent.fit()

print('Evaluating...')
ccresults, ccscores, ccytrue, ccpred, cc_original_row_id = ccagent.evaluate()
ccagent.save()

print()
print('=' * 60)
print('CREDIT CARD IF RESULTS')
print('=' * 60)
for k, v in ccresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')


Fitting Credit Card IF...
Best contamination: 0.001, tune F1: 0.5161, threshold: -0.008910
Evaluating...
Saved CC artifact.

CREDIT CARD IF RESULTS
  dataset                  : creditcard
  protocol                 : strict point-wise holdout (20% test)
  precision                : 0.464000
  recall                   : 0.591837
  f1                       : 0.520179
  rocauc                   : 0.959610
  prauc                    : 0.506539
  threshold                : -0.008910
  n_anomalies_true         : 98
  n_anomalies_pred         : 125


In [5]:
# Cell 6 — Export for Coordinator (Credit Card only)

exported = {}
summaryrows = []

# --- Credit Card ---
ccbundle = {
    'dataset': 'creditcard',
    'model': 'isolationforest',
    'protocol': 'strict point-wise holdout',
    'entities': {
        'creditcard': {
            'entityid': 'creditcard',
            'scoresfull': ccscores,
            'yfull': ccytrue,
            'predfull': ccpred,
            'rowid': np.arange(len(ccytrue), dtype=np.int64),
            'originalrowid': cc_original_row_id,
            'threshold': float(ccagent.threshold),
        }
    }
}

ccpath = os.path.join(PREDICTIONSDIR, 'ifcreditcardstrictpointwise.joblib')
joblib.dump(ccbundle, ccpath)
exported['creditcard'] = ccpath
summaryrows.append({
    'dataset': 'creditcard',
    **{k: v for k, v in ccresults.items() if k != 'dataset'},
})
print('Saved', ccpath)

# --- Summary CSV ---
summary = pd.DataFrame(summaryrows)
summarypath = os.path.join(PREDICTIONSDIR, 'ifstrictsummary.csv')
summary.to_csv(summarypath, index=False)
print('Saved', summarypath)
display(summary)

# --- Manifest ---
manifest = {
    'runid': RUNID,
    'driveroot': DRIVEROOT,
    'notebooktag': NOTEBOOKTAG,
    'modelfamily': 'isolationforest',
    'exportprotocol': 'ensembleexportv2',
    'artifactsdir': ARTIFACTDIR,
    'predictionsdir': PREDICTIONSDIR,
    'exports': exported,
    'summarycsv': summarypath,
    'creditcard_originalrowid_included': True,
    'datasets': ['creditcard'],
}
manifestpath = os.path.join(PREDICTIONSDIR, 'ifmanifest.json')
with open(manifestpath, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Saved', manifestpath)

print()
print('Coordinator-ready files in:', PREDICTIONSDIR)
print('Done.')

Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions/ifcreditcardstrictpointwise.joblib
Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions/ifstrictsummary.csv


,dataset,protocol,precision,recall,f1,rocauc,prauc,threshold,n_anomalies_true,n_anomalies_pred
0,creditcard,strict point-wise holdout (20% test),0.464,0.591837,0.520179,0.95961,0.506539,-0.00891,98,125


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions/ifmanifest.json

Coordinator-ready files in: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions
Done.


In [6]:
# Cell 7 — Final Summary

print('=' * 70)
print('ISOLATION FOREST FINAL SUMMARY')
print('=' * 70)

print()
print('CREDIT CARD:')
for k, v in ccresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

print()
print('All exports saved to:', PREDICTIONSDIR)
print('Done.')

ISOLATION FOREST FINAL SUMMARY

CREDIT CARD:
  dataset                  : creditcard
  protocol                 : strict point-wise holdout (20% test)
  precision                : 0.464000
  recall                   : 0.591837
  f1                       : 0.520179
  rocauc                   : 0.959610
  prauc                    : 0.506539
  threshold                : -0.008910
  n_anomalies_true         : 98
  n_anomalies_pred         : 125

All exports saved to: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/iforeststrict/predictions
Done.


In [7]:
# DEBUG — Isolation Forest contribution analysis
print('=== ISOLATION FOREST CONTRIBUTION ANALYSIS ===')
print()
print('IF ROLE IN ENSEMBLE:')
print('  - CC F1=0.52: moderate — catches ~59% of frauds')
print('  - High ROC-AUC (0.96): good anomaly ranking, threshold-sensitive')
print()
print('COMPLEMENTARITY with other models:')
print("  - IF uses random feature subspaces (different from ECOD's marginal approach)")
print('  - IF captures feature interactions via random partitioning')
print('  - Adds diversity via different inductive bias (isolation-based vs boosting)')
print()
print('In the coordinator, IF gets low CC weight via F1^2 weighting:')
print(f'  Weight factor: {0.52**2:.4f} (vs XGBoost {0.86**2:.4f})')
print('  Low individual F1 but contributes to ensemble diversity.')

=== ISOLATION FOREST CONTRIBUTION ANALYSIS ===

IF ROLE IN ENSEMBLE:
  - CC F1=0.52: moderate — catches ~59% of frauds
  - High ROC-AUC (0.96): good anomaly ranking, threshold-sensitive

COMPLEMENTARITY with other models:
  - IF uses random feature subspaces (different from ECOD's marginal approach)
  - IF captures feature interactions via random partitioning
  - Adds diversity via different inductive bias (isolation-based vs boosting)

In the coordinator, IF gets low CC weight via F1^2 weighting:
  Weight factor: 0.2704 (vs XGBoost 0.7396)
  Low individual F1 but contributes to ensemble diversity.
